#Majority Vote

## 4o

In [ ]:
import openai
import pandas as pd
import requests
import time

df = pd.read_csv("your_reddit_file.csv")

api_key = ""

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

def classify_question_4o(question):
    prompt = f"""
      We need to filter out certain types of questions that do not provide enough context or are too rigidly technical. The filtering process will follow these guidelines:

      Questions to Filter Out:
      1.Too Short Questions: any question that contains fewer than 40 words will be excluded, as it likely lacks sufficient context for meaningful analysis.
      Example: "Is milk good or bad for you?" (Lacks details or context)

      2. Technical or Fact-based Questions: questions that seek deterministic, factual answers based on established scientific standards or  have a clear, definitive answer in science or medicine will be filtered out.
      Example: "I've seen used swimsuits on Ebay and in thrift stores. I've also heard that buying used is a great way to save money and prevent waste. I'm not sure if a wearing someone else's swimsuit is a good idea, even if I were to I wash it in hot, soapy water. Is it hygienic to buy used swimsuits?"

      We are looking for questions that:
      1.Involve personal decision-making and multiple perspectives or allow for multiple solutions or strategies. The best questions are those where different people, backgrounds, or personalities might approach the problem differently and require advice rather than a single correct answer.

      Example: "I recently (about one month ago) met a guy on-line and we have been BF, GF for about three weeks now. Every one is worried about what will happen when we finally see each other for the first time. I just want to meet him. I feel as if I have known him for all my life. We talk all the time and he always tells me that he loves me. How can I be sure that what he says is what he really means?" (Different solutions may work for different people.)

      2. Provide enough context for meaningful discussion. The question should include some background information to help understand the situation, rather than being too vague.

      Example: "As a 21 year old, I recognize that I'm far from being completely emotionally developed. But, I also recognize that I'm way behind others my age. I think I am too emotionally sensitive. Things people say or do really affect me. Whether I care about the person or not, I always have extreme emotional episodes after others express their feelings or opinions about me. If what they express is derogatory, I get very upset. If it's positive, I get very happy. And, I absolutely cannot deal with rejection. I want to be able to just ignore what others think and just deal with what I think. How can I achieve that goal?"

      Now, the question is: {question}
      Respond only with "Yes" or "No".
    """

    try:
        payload = {
          "model": "gpt-4o",
          "messages": [

            {
              "role": "user",
              "content": [

                {
                  "type": "text",
                  "text":prompt
                }

              ]
            }
          ],
          "max_tokens": 4096,
          "temperature": 0
        }

        response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

        text=response.json()["choices"][0]["message"]["content"]
        print(text)
        return text

    except Exception as e:
        print(f"Error processing question: {question}\nError: {e}")
        return "Error"


df["Filter Result"] = df["scenario"].apply(lambda q: classify_question_4o(q) if isinstance(q, str) else "Error")

df.to_csv("reddit_scenario_filtered_questions_comp.csv", index=False)


##Qwen

In [ ]:
from together import Together
import os
os.environ["TOGETHER_API_KEY"] = ""
client = Together()
df = pd.read_csv("your_reddit_file.csv")

def classify_question_qwen(question):
    prompt = f"""
      We need to filter out certain types of questions that do not provide enough context or are too rigidly technical. The filtering process will follow these guidelines:

      Questions to Filter Out:
      1.Too Short Questions: any question that contains fewer than 40 words will be excluded, as it likely lacks sufficient context for meaningful analysis.
      Example: "Is milk good or bad for you?" (Lacks details or context)

      2. Technical or Fact-based Questions: questions that seek deterministic, factual answers based on established scientific standards or  have a clear, definitive answer in science or medicine will be filtered out.
      Example: "I've seen used swimsuits on Ebay and in thrift stores. I've also heard that buying used is a great way to save money and prevent waste. I'm not sure if a wearing someone else's swimsuit is a good idea, even if I were to I wash it in hot, soapy water. Is it hygienic to buy used swimsuits?"

      We are looking for questions that:
      1.Involve personal decision-making and multiple perspectives or allow for multiple solutions or strategies. The best questions are those where different people, backgrounds, or personalities might approach the problem differently and require advice rather than a single correct answer.

      Example: "I recently (about one month ago) met a guy on-line and we have been BF, GF for about three weeks now. Every one is worried about what will happen when we finally see each other for the first time. I just want to meet him. I feel as if I have known him for all my life. We talk all the time and he always tells me that he loves me. How can I be sure that what he says is what he really means?" (Different solutions may work for different people.)

      2. Provide enough context for meaningful discussion. The question should include some background information to help understand the situation, rather than being too vague.

      Example: "As a 21 year old, I recognize that I'm far from being completely emotionally developed. But, I also recognize that I'm way behind others my age. I think I am too emotionally sensitive. Things people say or do really affect me. Whether I care about the person or not, I always have extreme emotional episodes after others express their feelings or opinions about me. If what they express is derogatory, I get very upset. If it's positive, I get very happy. And, I absolutely cannot deal with rejection. I want to be able to just ignore what others think and just deal with what I think. How can I achieve that goal?"

      Now, the question is: {question}
      Respond only with "Yes" or "No".
    """

    try:
        response = client.chat.completions.create(
          model="Qwen/Qwen2.5-72B-Instruct-Turbo",
          messages=[{"role": "user", "content": prompt}],
          temperature=0
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content

    except Exception as e:
        print(f"Error processing question: {question}\nError: {e}")
        return "Error"


df["Filter Result"] = df["scenario"].apply(lambda q: classify_question_qwen(q) if isinstance(q, str) else "Error")

df.to_csv("reddit_scenario_filtered_questions_qwen_comp.csv", index=False)




In [ ]:
import pandas as pd

df1 = pd.read_csv("reddit_scenario_filtered_questions_qwen_comp.csv")
df2 = pd.read_csv("reddit_scenario_filtered_questions_comp.csv")

df1['Filter Result'] = df1['Filter Result'].str.lower().str.strip('.')
df2['Filter Result'] = df2['Filter Result'].str.lower().str.strip('.')


yes_rows_1 = set(df1[df1['Filter Result'] == 'yes'].index)
yes_rows_2 = set(df2[df2['Filter Result'] == 'yes'].index)

all_yes = yes_rows_1 & yes_rows_2
part_yes = (yes_rows_1 ^ yes_rows_2)
no_yes = set(df1.index) - (yes_rows_1 | yes_rows_2)

all_list = list(all_yes)
part_list = list(part_yes)
no_list = list(no_yes)

meta_descriptions = df1.loc[part_list, ['scenario']]
print(meta_descriptions)

## deepseek

In [ ]:
def classify_question_deepseek(question):
    prompt = f"""
      We need to filter out certain types of questions that do not provide enough context or are too rigidly technical. The filtering process will follow these guidelines:

      Questions to Filter Out:
      1.Too Short Questions: any question that contains fewer than 40 words will be excluded, as it likely lacks sufficient context for meaningful analysis.
      Example: "Is milk good or bad for you?" (Lacks details or context)

      2. Technical or Fact-based Questions: questions that seek deterministic, factual answers based on established scientific standards or  have a clear, definitive answer in science or medicine will be filtered out.
      Example: "I've seen used swimsuits on Ebay and in thrift stores. I've also heard that buying used is a great way to save money and prevent waste. I'm not sure if a wearing someone else's swimsuit is a good idea, even if I were to I wash it in hot, soapy water. Is it hygienic to buy used swimsuits?"

      We are looking for questions that:
      1.Involve personal decision-making and multiple perspectives or allow for multiple solutions or strategies. The best questions are those where different people, backgrounds, or personalities might approach the problem differently and require advice rather than a single correct answer.

      Example: "I recently (about one month ago) met a guy on-line and we have been BF, GF for about three weeks now. Every one is worried about what will happen when we finally see each other for the first time. I just want to meet him. I feel as if I have known him for all my life. We talk all the time and he always tells me that he loves me. How can I be sure that what he says is what he really means?" (Different solutions may work for different people.)

      2. Provide enough context for meaningful discussion. The question should include some background information to help understand the situation, rather than being too vague.

      Example: "As a 21 year old, I recognize that I'm far from being completely emotionally developed. But, I also recognize that I'm way behind others my age. I think I am too emotionally sensitive. Things people say or do really affect me. Whether I care about the person or not, I always have extreme emotional episodes after others express their feelings or opinions about me. If what they express is derogatory, I get very upset. If it's positive, I get very happy. And, I absolutely cannot deal with rejection. I want to be able to just ignore what others think and just deal with what I think. How can I achieve that goal?"

      Now, the question is: {question}
      Respond only with "Yes" or "No".
    """

    try:
        response = client.chat.completions.create(
          model="deepseek-ai/DeepSeek-V3",
          messages=[{"role": "user", "content": prompt}],
          temperature=0
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content

    except Exception as e:
        print(f"Error processing question: {question}\nError: {e}")
        return "Error"

df.loc[part_list, "Filter Result"] = df.loc[part_list, "scenario"].apply(lambda q: classify_question_deepseek(q) if isinstance(q, str) else "Error")
df.loc[all_list, "Filter Result"] = "Yes"
df.loc[no_list, "Filter Result"] = "No"

df.to_csv("reddit_scenario_filtered_questions_mjvt_comp.csv", index=False)